# 01 -- Corpus Exploration

This notebook loads the synthetic dialect corpus, displays per-variety statistics,
and visualises sample sentences from each of the 8 Spanish dialect varieties.

**Key modules:** `corpus.synthetic.generator`, `corpus.synthetic.templates`, `constants`

In [ ]:
import pandas as pd
from eigendialectos.constants import DialectCode, DIALECT_NAMES, DIALECT_REGIONS
from eigendialectos.corpus.synthetic.generator import SyntheticGenerator
from eigendialectos.corpus.synthetic.templates import DIALECT_TEMPLATES, list_templates

## Generate Synthetic Corpus

The `SyntheticGenerator` transforms neutral Peninsular Spanish sentences through
dialect-specific templates (lexical, morphological, phonological rules).

In [ ]:
gen = SyntheticGenerator(seed=42)
corpus = gen.generate_all(n_per_dialect=200)

print(f"Base sentence bank size: {gen.base_sentence_count}")
print(f"Dialects generated: {len(corpus)}")
print(f"Total samples: {sum(len(s.samples) for s in corpus.values())}")

## Per-Variety Statistics

In [ ]:
rows = []
for code, corpus_slice in sorted(corpus.items(), key=lambda x: x[0].value):
    stats = corpus_slice.stats
    rows.append({
        "Code": code.value,
        "Name": DIALECT_NAMES.get(code, "?"),
        "Region": DIALECT_REGIONS.get(code, "?"),
        "Count": stats["count"],
        "Avg Length": f"{stats['avg_length']:.1f}",
        "Min Length": stats["min_length"],
        "Max Length": stats["max_length"],
    })

pd.DataFrame(rows)

## Sample Sentences by Dialect

Show the first 3 transformed sentences alongside the neutral base for each variety.

In [ ]:
for code in sorted(corpus.keys(), key=lambda c: c.value):
    name = DIALECT_NAMES.get(code, code.value)
    print(f"\n{'='*70}")
    print(f"{name} ({code.value})")
    print(f"{'='*70}")
    for sample in corpus[code].samples[:3]:
        base = sample.metadata.get("base_sentence", "")
        print(f"  BASE: {base}")
        print(f"  DIAL: {sample.text}")
        print(f"  conf: {sample.confidence}")
        print()

## Transformation Template Summary

Number of rules per category for each dialect.

In [ ]:
template_rows = []
for code in list_templates():
    tmpl = DIALECT_TEMPLATES[code]
    template_rows.append({
        "Dialect": code.value,
        "Lexical substitutions": len(tmpl.lexical),
        "Morphological rules": len(tmpl.morphological),
        "Phonological rules": len(tmpl.phonological),
        "Pragmatic markers": len(tmpl.pragmatic_markers),
    })

pd.DataFrame(template_rows).set_index("Dialect")

## Text Length Distribution

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
for code in sorted(corpus.keys(), key=lambda c: c.value):
    lengths = [len(s.text) for s in corpus[code].samples]
    ax.hist(lengths, bins=20, alpha=0.5, label=code.value)

ax.set_xlabel("Text length (characters)")
ax.set_ylabel("Count")
ax.set_title("Text Length Distribution by Dialect")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## Confidence Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
data = []
labels = []
for code in sorted(corpus.keys(), key=lambda c: c.value):
    confs = [s.confidence for s in corpus[code].samples]
    data.append(confs)
    labels.append(code.value)

ax.boxplot(data, labels=labels)
ax.set_ylabel("Confidence")
ax.set_title("Transformation Confidence by Dialect")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()